# Differentiation Methods in Deep Learning

Before diving into PyTorch's `autograd`, it is essential to understand the three primary ways computers calculate derivatives and why traditional methods fail when training modern neural networks.

---


# 1. Numerical Differentiation

### Theory
Numerical differentiation approximates the derivative of a function using the definition of a limit with a very small step size h (finite differences):

\[f'(x) \approx \frac{f(x + h) - f(x)}{h}\]

**Why it fails for Neural Networks:**

1. **Computational Complexity (O(N)):** To compute the gradient for a network with N parameters, you must run the forward pass N + 1 times (perturbing one weight at a time). For a model with 1 billion parameters, a single update step requires 1 billion forward passes.

2. **Numerical Instability:** Floating-point arithmetic creates a trade-off. If h is too large, the **truncation error** is high. If h is too small, **round-off error** (catastrophic cancellation) occurs because f(x+h) and f(x) become nearly identical in memory.


In [10]:
import numpy as np
import torch
import time

In [8]:
# 1. Demonstrating Round-off Error (Precision Loss)
def f(x):
    return x**2

# Gradient at : 
x = 3.0
exact_grad = 6.0

print(" ---- Precision Loss ----")

def numeric_limit(x,h):
    return ( f(x+h) - f(x) ) / h

h = 0.02
for i in range(10):
    numeric_grad = numeric_limit(x, h)
    print(f"h = {h:.2e} | Approx Grad = {numeric_grad:<20} | Error = {abs(exact_grad - numeric_grad):.2e} ")
    h *= 0.1



 ---- Precision Loss ----
h = 2.00e-02 | Approx Grad = 6.020000000000003    | Error = 2.00e-02 
h = 2.00e-03 | Approx Grad = 6.001999999999619    | Error = 2.00e-03 
h = 2.00e-04 | Approx Grad = 6.000200000002565    | Error = 2.00e-04 
h = 2.00e-05 | Approx Grad = 6.000020000040961    | Error = 2.00e-05 
h = 2.00e-06 | Approx Grad = 6.000001999240111    | Error = 2.00e-06 
h = 2.00e-07 | Approx Grad = 6.000000203343347    | Error = 2.03e-07 
h = 2.00e-08 | Approx Grad = 5.999999963535172    | Error = 3.65e-08 
h = 2.00e-09 | Approx Grad = 6.000000496442223    | Error = 4.96e-07 
h = 2.00e-10 | Approx Grad = 6.000000496442222    | Error = 4.96e-07 
h = 2.00e-11 | Approx Grad = 6.000000496442222    | Error = 4.96e-07 


In [14]:
# 2. Demonstrating Computational Bottleneck
print("------ speed Bottleneck ------- ")
weights = np.random.randn(10_000)
h = 1e-5

def forward_pass(w):
    return np.sum(np.sin(w) ** 2)

start = time.time()
grads = np.zeros_like(weights)
base_loss = forward_pass(weights)

# Must loop over EVERY parameters individually 
for i in range(len(weights)):
    w_perturbed = weights.copy()
    w_perturbed[i] += h 
    grads[i] = (forward_pass(w_perturbed) - base_loss) / h

print(f"Time for 10,000 Parameters : {time.time() - start:.4f} sec ")

------ speed Bottleneck ------- 
Time for 10,000 Parameters : 0.3217 sec 


# 2. Symbolic Differentiation

### Theory
Symbolic differentiation manipulates mathematical expressions algebraically using calculus rules (Product Rule, Chain Rule, etc.) to produce a new closed-form mathematical formula for the derivative. Tools like SymPy or Mathematica use this approach.

**Why it fails for Neural Networks:**

1. **Expression Swell:** Repeatedly applying the product and chain rules to deeply nested functions (like a 100-layer neural network) causes the symbolic equation to grow exponentially in size, quickly exhausting system RAM.

2. **Control Flow Incompatibility:** Neural networks rely heavily on programming constructs like `if/else` statements, `for/while` loops, and recursion. Symbolic math engines cannot easily differentiate through dynamic Python control flow.




In [18]:
import sympy as sp

# 1. Demonstrating Expression Swell 
x = sp.Symbol('x')
expr = sp.sin(x)


print(" ---- Expression Swell ----- ")
# Nesting a simple function just 4 times (like a 4-layer network)
for layer in range(1,5):
    expr = sp.sin(expr * x)
    deriv = sp.diff(expr, x)
    
    print(f"Layer {layer} derivative length : {len(str(deriv))} characters")
    if layer == 3:
        print(f"Layer 3 formula preview : {str(deriv)[:120]}.......\n")

 ---- Expression Swell ----- 
Layer 1 derivative length : 33 characters
Layer 2 derivative length : 74 characters
Layer 3 derivative length : 129 characters
Layer 3 formula preview : (x*(x*(x*cos(x) + sin(x))*cos(x*sin(x)) + sin(x*sin(x)))*cos(x*sin(x*sin(x))) + sin(x*sin(x*sin(x))))*cos(x*sin(x*sin(x*.......

Layer 4 derivative length : 198 characters


In [19]:
# 2. Demonstrating Control Flow Failure
print(" --- Control Flow Failure ---- ")
def relu(val):
    if val>0: # python boolean evaluation fails on symbolic variable
        return val
    return 0

try:
    relu(x)
except TypeError as e:
    print(f"Symbolic Engin Error on if statement : {e}")

 --- Control Flow Failure ---- 
Symbolic Engin Error on if statement : cannot determine truth value of Relational: x > 0


# 3. Automatic Differentiation (Autograd)

### Theory
Automatic Differentiation (AD) bridges the gap between numerical and symbolic methods. Instead of approximating values or building massive algebraic formulas, AD decomposes any complex program into a sequence of **elementary operations** (`+`, `-`, `*`, `/`, `sin`, `exp`, etc.) whose derivatives are already known.

As the forward pass runs, AD records these operations in a **Directed Acyclic Graph (DAG)** (computational graph). It then applies the **Chain Rule** numerically to intermediate values to compute exact gradients up to machine precision.

**Two Modes of Autograd:**

* **Forward-Mode AD:** Computes derivatives from inputs to outputs. Efficient when `inputs << outputs`.

* **Reverse-Mode AD (Backpropagation):** Computes derivatives from outputs to inputs. Computes the gradient of a single scalar loss with respect to **millions of weights in just one backward pass** (O(1) passes).




In [ ]:
# 1. Exact Precision & Speed on Large Networks
weights = torch.randn(10_100, requires_grad=True)

start = time.time()

# Forward pass builds the dynamic computational graph
loss = torch.sum(torch.sin(weights) ** 2)

# Single backward pass computes all 10,000 exact gradient simultaneously
loss.backward()
print(f"Pytorch Autograd time for 10,000 params: {time.time() - start:.6f} sec")



Pytorch Autograd time for 10,000 params: 0.032054 sec


In [22]:
# 2. Seamless Control flow Support (Define-by-Run)
x = torch.tensor(-3.0, requires_grad=True)

def dynamic_network(val):
    # Standard Python loop and conditionals works natively
    for _ in range(4):
        if val < 0:
            val = val *2.0
        else:
            val = val + 5.0
    return val ** 2

out = dynamic_network(x)
out.backward()
print(f"Exact gradient through dynamic control flow : {x.grad}")

Exact gradient through dynamic control flow : -1536.0
